# Required Libraries and Functions

In [0]:
import requests, json
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lit, max, expr
from pyspark.sql.types import DateType

In [0]:
dbutils.widgets.text("volume_path", "/Volumes/workspace/taxi_weather/datasets/")

base_path = dbutils.widgets.get("volume_path")

# Load data from DBFS

In [0]:
borough_file_path = (f'{base_path}borough_coord.json')

with open(borough_file_path, "r") as f:
    coord_data = json.load(f)

dates_file_path = f"{base_path}pipeline_dates.json"

with open(dates_file_path, "r") as f:
    date_params = json.load(f)

start_date = date_params["start_date"]
end_date = date_params["end_date"]

# Get Hourly Weather Data using Open-Meteo's Historical API

In [0]:
weather_data = []

for borough_name, coords in coord_data.items():
    lat = coords["latitude"]
    lon = coords["longitude"]

    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}"
        f"&longitude={lon}"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        f"&timezone=America%2FNew_York"
        f"&hourly=temperature_2m,precipitation,snowfall,weather_code"
    )


    response = requests.get(url, timeout=10)
    try:
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to fetch weather data for {borough_name}. Error: {e}")

    hourly_data = response.json()["hourly"]

    times = hourly_data["time"]
    temps = hourly_data["temperature_2m"]
    precip = hourly_data["precipitation"]
    snow = hourly_data["snowfall"]
    codes = hourly_data["weather_code"]

    for t, temp, p, s, c in zip(times, temps, precip, snow, codes):
        weather_data.append(
            {
                "borough": borough_name,
                "weather_datetime_str": t,
                "temperature_2m": temp,
                "precipitation": p,
                "snowfall": s,
                "weather_code": c,
            }
            )

## Add relevant columns

In [0]:
df = spark.createDataFrame(weather_data)

bronze_weather = (df
                  .withColumn("weather_datetime", to_timestamp(col("weather_datetime_str")))
                  .withColumn("source_file_name", lit("Open-Meteo Historical Weather API"))
                  .withColumn("ingest_time", current_timestamp())
                  .drop("weather_datetime_str")
                  )

# Write to Delta Tables

In [0]:
(
    bronze_weather.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.taxi_weather.bronze_weather")
)